[Reference](https://pub.towardsai.net/fine-tuning-functiongemma-from-75-to-100-accuracy-in-3-minutes-d26096d498be)

# Software Setup

In [1]:
# Clone repository
!git clone https://github.com/jageenshukla/functiongemma-finetuning-lora.git
!cd functiongemma-finetuning-lora

# Setup virtual environment
!python3 -m venv venv
!source venv/bin/activate  # On Windows: venv\Scripts\activate

# Install dependencies
!pip install -r requirements.txt

# Login to HuggingFace
!python -c "from huggingface_hub import login; login()"

# Define Your Functions

In [2]:
# config/music_functions.py

MUSIC_FUNCTIONS = [
    {
        "type": "function",
        "function": {
            "name": "play_song",
            "description": "Play a specific song by name or artist",
            "parameters": {
                "type": "object",
                "properties": {
                    "song_name": {
                        "type": "string",
                        "description": "Name of the song to play"
                    },
                    "artist": {
                        "type": "string",
                        "description": "Artist name (optional)"
                    }
                },
                "required": ["song_name"]
            }
        }
    },
    # Add more functions...
]

# Create Training Examples

In [3]:
# data/four_func_examples.py

EXAMPLES = [
    {
        "user_input": "Play Bohemian Rhapsody",
        "function_name": "play_song",
        "arguments": {"song_name": "Bohemian Rhapsody"}
    },
    {
        "user_input": "I want to hear Hotel California by Eagles",
        "function_name": "play_song",
        "arguments": {
            "song_name": "Hotel California",
            "artist": "Eagles"
        }
    },
    # ... 98 more examples (25 per function)
]

# Format the Dataset

In [4]:
# scripts/generate_4func_dataset.py

from transformers import AutoTokenizer
from datasets import Dataset
import json

tokenizer = AutoTokenizer.from_pretrained("google/functiongemma-270m-it")

def format_example(example):
    """Convert raw example to FunctionGemma format"""
    messages = [
        {
            "role": "user",
            "content": example["user_input"]
        },
        {
            "role": "assistant",
            "tool_calls": [{
                "type": "function",
                "function": {
                    "name": example["function_name"],
                    "arguments": example["arguments"]  # Pass dict!
                }
            }]
        }
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tools=MUSIC_FUNCTIONS,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

# Process and save
formatted_data = [format_example(ex) for ex in EXAMPLES]
# Split 80/20 and save as JSONL files

```
python scripts/generate_4func_dataset.py
```

# Configure LoRA Fine-Tuning

Full Fine-Tuning:
- Parameters Updated: 271M (100%)
- Memory Usage: High
- Training Time: Slow
- Quality: 100%
LoRA:
- Parameters Updated: 3.8M (1.4%)
- Memory Usage: Low
- Training Time: Fast
- Quality: 99%

In [5]:
# scripts/train_4func.py

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,           # LoRA scaling
    target_modules=[         # 🚨 CRITICAL: All 7 modules required
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Check adapter size
```
ls -lh models/*/final/adapter_model.safetensors
```

# Train the Model

In [6]:
# scripts/train_4func.py (continued)

from transformers import AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    "google/functiongemma-270m-it",
    device_map="cpu",
    torch_dtype="auto"
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Training configuration
training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch"
)

# CRITICAL: Define formatting function
def formatting_func(example):
    return example["text"]

# Create trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    formatting_func=formatting_func  # REQUIRED!
)

# Train!
trainer.train()

# Test Your Model

In [7]:
# scripts/local_demo.py

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_PATH = "models/music-4func-*/final"

# Load model
base_model = AutoModelForCausalLM.from_pretrained(
    "google/functiongemma-270m-it",
    device_map="cpu"
)

model = PeftModel.from_pretrained(base_model, MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Test
def test_command(user_input):
    messages = [{"role": "user", "content": user_input}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tools=MUSIC_FUNCTIONS,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0])

# Test all cases
result = test_command("Play Bohemian Rhapsody")
print(result)

# Production Deployment

In [8]:
from transformers import AutoModelForCausalLM
from peft import PeftModel
from flask import Flask, request, jsonify

app = Flask(__name__)

class MusicAssistant:
    def __init__(self):
        base_model = AutoModelForCausalLM.from_pretrained(
            "google/functiongemma-270m-it"
        )
        self.model = PeftModel.from_pretrained(
            base_model,
            "your-username/music-4func"
        )
        self.model = self.model.merge_and_unload()

    def parse_command(self, user_input):
        # Generate and parse function call
        return function_call

assistant = MusicAssistant()

@app.route('/parse', methods=['POST'])
def parse():
    command = request.json.get('command')
    result = assistant.parse_command(command)
    return jsonify(result)